# Lab 05 — GraphRAG: retrieval that follows relationships

Original OfferReady lab. Build a tiny knowledge graph, traverse it to gather a
relevant **subgraph**, serialize it, and ground an answer over connected facts.
Dependency-free so it runs anywhere. Pairs with **Study Guide Ch7 (Graph DBs)**.

**Why:** pure vector RAG retrieves similar chunks independently; it struggles when
the answer is spread across *relationships* ("which suppliers connect to a flagged
account, two hops out?"). GraphRAG follows the edges.

## 1. A tiny graph (nodes + typed edges)

In [ ]:
# nodes: id -> (label, props);  edges: list of (src, rel, dst)
nodes = {
    "c1": ("Customer", {"name": "Acme"}),
    "o1": ("Order", {"total": 500}),
    "p1": ("Product", {"name": "Widget"}),
    "s1": ("Supplier", {"name": "Globex", "flagged": True}),
}
edges = [
    ("c1", "PLACED", "o1"),
    ("o1", "CONTAINS", "p1"),
    ("p1", "SUPPLIED_BY", "s1"),
]

## 2. Traverse to a relevant subgraph

Breadth-first from a start node up to N hops — the graph equivalent of "retrieve
top-k", but following relationships instead of vector similarity.

In [ ]:
def neighbors(node_id):
    for s, rel, d in edges:
        if s == node_id:
            yield rel, d

def subgraph(start, max_hops=3):
    seen, frontier, collected = {start}, [start], []
    for _ in range(max_hops):
        nxt = []
        for nid in frontier:
            for rel, dst in neighbors(nid):
                collected.append((nid, rel, dst))
                if dst not in seen:
                    seen.add(dst); nxt.append(dst)
        frontier = nxt
    return collected

path = subgraph("c1")
for s, rel, d in path:
    print(f"{nodes[s][1].get('name', s)} -{rel}-> {nodes[d][1].get('name', d)}")

## 3. Serialize the subgraph + ground an answer

Turn the path into text the model can reason over, then ask a relationship
question. (LLM stubbed so it runs offline — swap in Lab 03's Converse call.)

In [ ]:
def serialize(path):
    lines = []
    for s, rel, d in path:
        sp, dp = nodes[s][1], nodes[d][1]
        lines.append(f"({nodes[s][0]}:{sp}) -{rel}-> ({nodes[d][0]}:{dp})")
    return "\n".join(lines)

def fake_llm(prompt):
    return "Acme is connected (via order → product) to supplier Globex, which is flagged."

question = "Is Acme connected to any flagged supplier, and how?"
prompt = (
    "Answer ONLY from the graph facts. Cite the path.\n\n"
    f"<graph>\n{serialize(subgraph('c1'))}\n</graph>\n\nQuestion: {question}"
)
print(prompt)
print("\n--- answer ---")
print(fake_llm(prompt))

## Mapping to production

- Replace the dict graph with **Neo4j** and the traversal with a **Cypher** query
  (e.g. `MATCH (c:Customer)-[*1..3]->(s:Supplier {flagged:true})`).
- Combine with vector RAG: vector search to find entry-point entities, then graph
  traversal to expand the neighborhood.
- Serialize the subgraph into the prompt and require the model to cite the path.

See the Study Guide, Chapter 7 (Graph databases for AI).